# 2026F - San Jose State University - CMPE-257 - project - Chris Carl

## Metadata
<div style="display: inline-block">
<!-- get the table rendered to the left rather than default center, the spaces after the raw html are important for markdown https://stackoverflow.com/a/78525025 -->

| Key | Value |
| :--- | :--- |
| author | Chris Carl(mailto:chris.carl@sjsu.edu) |
| date | 2026-11-13 |
| dataset | [AmesHousing.txt, save as CSV](https://jse.amstat.org/v19n3/decock/AmesHousing.txt) © 2015 American Statistical Association |

<!-- get the table rendered to the left rather than default center, the spaces after the raw html are important for markdown https://stackoverflow.com/a/78525025 -->
</div>

<!--
commands:
    ipynb CMPE-257/assignments/project/project.ipynb  # chriscarl.tools.documents
-->


# Abstract

This is my submission of project of CMPE-257 at SJSU instructed by Dr. Zara Hajihashemi in Fall 2026.

Explain the executive summary and findings. You HAVE TO. this is the first thing they see.


# TOC


<!-- # SUGGESTED TOC ORGANIZATION

# Project
# Abstract
# Table of Contents
# Question
# Answer
# Solution
## Background
## The Plan
## Code - Environment
## Code - Setup
## Code - Functions
## Code - Main
# Post Mortem
## Caveats
## Improvements
## Replication
# Appendix
## Criterion
## Changelog
-->

# Solution


## Code - Setup

The following is boilerplate I've accumulated over the years and can be safely disregarded by the reader until it becomes relevant as part of deeper inquiry.


In [1]:
# stdlib imports
import __main__
import os
import sys
import pprint
import importlib
import subprocess
# hyper-critical imports
try:
    from IPython import get_ipython
    IPYTHON_SHELL = get_ipython()  # None if running in vanilla python
except ImportError:
    IPYTHON_SHELL = None

SCRIPT_FILEPATH = ''
SCRIPT_FILENAME = ''
if IPYTHON_SHELL:
    SCRIPT_DIRPATH = str(IPYTHON_SHELL.run_line_magic('pwd', ' # %pwd is a "magic" command  https://ipython.readthedocs.io/en/stable/interactive/magics.html'))
    if hasattr(__main__, '__vsc_ipynb_file__'):  # vscode
        SCRIPT_FILEPATH = __main__.__vsc_ipynb_file__
    elif hasattr(__main__, '__session__'):  # localhost:8888/notebooks
        SCRIPT_FILEPATH = __main__.__session__
else:
    SCRIPT_FILEPATH = __file__
    SCRIPT_DIRPATH = os.path.dirname(SCRIPT_FILEPATH)

SCRIPT_FILENAME = os.path.splitext(os.path.basename(SCRIPT_FILEPATH))[0]

pprint.pprint(dict(dir=SCRIPT_DIRPATH, fp=SCRIPT_FILEPATH, fn=SCRIPT_FILENAME), indent=2)

AUTO_INSTALL = False
packages = {}  # {'Werkzeug': 'werkzeug'}  # package name: module name
modules = ['matplotlib', 'pandas', 'ipykernel', 'ipympl', 'ipywidgets']
packages.update({m: m for m in modules})
package, module = '', ''
install_string = ' '.join(packages)
try:
    for package, module in packages.items():
        importlib.import_module(module)
except ImportError:
    print(f'"pip install {package}"', file=sys.stderr)
    if AUTO_INSTALL:
        %pip install {install_string}
    else:
        raise

# third party imports
import matplotlib.pyplot as plt

EXES = {
    'pandoc': {'exists': False},
    'latex': {'exists': False, 'win32': 'miktex', 'linux': 'texlive'},
}

for exe, data in EXES.items():
    if IPYTHON_SHELL:
        if sys.platform == 'win32':
            _res = !where.exe {exe}
            _exists = not _res[0].startswith('INFO')
        else:
            _res = !which {exe}
            _exists = bool(_res)
    else:
        if sys.platform == 'win32':
            _res = subprocess.check_output(f'where.exe {exe}', universal_newlines=True)
            _exists = not _res.startswith('INFO')
        else:
            _res = subprocess.check_output(f'which {exe}', universal_newlines=True)
            _exists = bool(_res)
    EXES[exe]['exists'] = _exists

print('pandoc:', EXES['pandoc']['exists'])
if not EXES['pandoc']['exists']:
    print('you must install pandoc!', file=sys.stderr)
    if sys.platform == 'win32':
        print('    choco install pandoc -y', file=sys.stderr)
    else:
        print('    sudo apt install pandoc -y', file=sys.stderr)
        print('    sudo yum install pandoc -y', file=sys.stderr)
        print('    dnf install pandoc -y', file=sys.stderr)
    raise RuntimeError()

print('latex:', EXES['latex']['exists'])
plt.rcParams.update(
    {
        'text.usetex': EXES['latex']['exists'],  # , 'font.family': 'Helvetica'  # requires latex installed
        'text.latex.preamble': r'\usepackage{amsmath}',  # https://stackoverflow.com/a/23856968  \text command
    }

)

# command to tell the notebook to plt.show() IN THE NOTEBOOK, otherwise you call plt.show()
%matplotlib inline
# command to tell the notebook to render interactable matplotlib, requires pip install ipympl
# %matplotlib widget

print('# pip install', install_string)
if not IPYTHON_SHELL:
    raise RuntimeError('Not sure how, but somehow this code is being run through a non-iPython environment!')


{ 'dir': 'c:\\Users\\chris\\src\\sjsu.CMPE-257.ml-social-bot-detection\\src',
  'fn': 'project',
  'fp': 'c:\\Users\\chris\\src\\sjsu.CMPE-257.ml-social-bot-detection\\src\\project.ipynb'}
pandoc: True
latex: True
# pip install matplotlib pandas ipykernel ipympl ipywidgets


# Setup

In [2]:
PROJECT_DIRECTORY = os.path.abspath(os.path.join(SCRIPT_DIRPATH, '..'))
DATASET_DIRECTORIES = [
    os.path.join(PROJECT_DIRECTORY, r'ignoreme\data\cresci-17'),
    os.path.join(PROJECT_DIRECTORY, r'ignoreme\data\twibot-22'),
]


In [3]:
import os
import zipfile

def unzip_file(zip_path, extract_to, password=None, print_progress=True):
    if not os.path.isfile(zip_path):
        raise FileNotFoundError(f"ZIP file not found: {zip_path}")
    os.makedirs(extract_to, exist_ok=True)
    try:
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            pwd = password.encode() if password else None
            for member in zip_ref.infolist():
                # Prevent Zip Slip vulnerability
                extracted_path = os.path.join(extract_to, member.filename)
                if member.filename.startswith('__MACOSX/._') or member.filename.split('/')[-1].startswith('._'):
                    continue

                if not os.path.realpath(extracted_path).startswith(os.path.realpath(extract_to)):
                    raise Exception(f"Unsafe file path detected: {member.filename}")

                if print_progress:
                    print(member)
                zip_ref.extract(member, path=extract_to, pwd=pwd)

                basename, ext = os.path.splitext(member.filename)
                if ext == '.zip':
                    unzip_file(os.path.join(extract_to, member.filename), os.path.join(extract_to, basename), password=password, print_progress=print_progress)
    except zipfile.BadZipFile:
        raise zipfile.BadZipFile(f"Invalid ZIP file: {zip_path}")
    except RuntimeError as e:
        if "password required" in str(e).lower():
            raise RuntimeError("This ZIP file is password-protected. Please provide a valid password.")
        else:
            raise


In [4]:
ZIPS = [
    os.path.join(PROJECT_DIRECTORY, r'ignoreme\data\cresci-17\datasets_full.csv.zip'),
]
for file in ZIPS:
    basename, _ = os.path.splitext(file)
    unzip_file(file, basename)


<ZipInfo filename='datasets_full.csv/' filemode='drwxr-xr-x' external_attr=0x10>
<ZipInfo filename='datasets_full.csv/crowdflower_results.csv.zip' compress_type=deflate filemode='-rw-r--r--' external_attr=0x4000 file_size=290528 compress_size=289584>
<ZipInfo filename='crowdflower_results.csv/' filemode='drwxr-xr-x' external_attr=0x4000>
<ZipInfo filename='crowdflower_results.csv/crowdflower_results_aggregated.csv' compress_type=deflate filemode='-rw-r--r--' external_attr=0x4000 file_size=617235 compress_size=107077>
<ZipInfo filename='__MACOSX/' filemode='drwxrwxr-x' external_attr=0x4000>
<ZipInfo filename='__MACOSX/crowdflower_results.csv/' filemode='drwxrwxr-x' external_attr=0x4000>
<ZipInfo filename='crowdflower_results.csv/crowdflower_results_contributors.csv' compress_type=deflate filemode='-rw-r--r--' external_attr=0x4000 file_size=24118 compress_size=8227>
<ZipInfo filename='crowdflower_results.csv/crowdflower_results_detailed.csv' compress_type=deflate filemode='-rw-r--r--' ex

In [5]:
import multiprocessing
import concurrent.futures
from typing import Callable, Any

MAX_WORKERS = multiprocessing.cpu_count() - 1

def parallel(funcs, args, kwargs, key=lambda args: args[0], max_workers=MAX_WORKERS):
    # type: (list[Callable], list[tuple], list[dict], Callable, int) -> list[tuple[Any, Any|Exception]]
    # https://docs.python.org/3/library/concurrent.futures.html
    N = len(funcs)
    if len(funcs) != len(args) or len(args) != len(kwargs):
        raise ValueError('need to provide same list amount of funcs, args, kwargs')

    res = []
    # NOTE: to run this, you need to use external files for the functions you have in mind
    with concurrent.futures.ProcessPoolExecutor(max_workers=max_workers) as executor:
        future_to_ret = {executor.submit(funcs[i], *args[i], **kwargs[i]): key(args[i]) for i in range(len(args))}
        for f, future in enumerate(concurrent.futures.as_completed(future_to_ret)):
            arg = future_to_ret[future]
            try:
                data = future.result()
            except Exception as exc:
                print(f'{f+1} / {N} - FAILURE! {arg}: {exc}', file=sys.stderr)
                res += [(arg, exc)]
            else:
                print(f'{f+1} / {N} - SUCCESS! {arg}: {data}', file=sys.stdout)
                res += [(arg, data)]
    return res


In [6]:
%%writefile data_to_pickle.py
import os
import pickle
import warnings
import pandas as pd

def data_to_pickle(filepath, pickle_filepath):
    ext = os.path.splitext(filepath)[1]
    if ext.lower() == '.json':
        df = pd.read_json(filepath, dtype=str, orient='records')
    elif ext.lower() == '.csv':
        df = pd.read_csv(filepath, dtype=str, low_memory=False)
    else:
        raise NotImplementedError(f'no parser implemented for {ext.lower()!r}')

    changed_columns = []
    for col in df.select_dtypes(include=['object'], exclude=['str']).columns:
        df[col] = df[col].astype('category')
        changed_columns.append((col, 'category'))

    for col in df.columns:
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                df[col] = pd.to_datetime(df[col])
                changed_columns.append((col, 'datetime'))
        except Exception:
            pass
        if df[col].nunique() < 100:
            df[col] = df[col].astype('category')
            changed_columns.append((col, 'category'))

    with open(pickle_filepath, 'wb') as f:
        pickle.dump(df, f)

    return changed_columns

Overwriting data_to_pickle.py


In [ ]:
import os
import glob
import pandas as pd
from data_to_pickle import data_to_pickle


funcs, args, kwargs = [], [], []
for d, dataset_directory in enumerate(DATASET_DIRECTORIES):
    globs = list(glob.glob(os.path.join(dataset_directory, '**', '*.csv'), recursive=True))
    globs += list(glob.glob(os.path.join(dataset_directory, '**', '*.json'), recursive=True))
    globs = sorted(globs, key=lambda pth: os.path.getsize(pth), reverse=True)  # complete in greatest to least
    for c, data_filepath in enumerate(globs):
        if not os.path.isfile(data_filepath):  # could be named datasets_full.csv...
            continue
        filename = os.path.splitext(data_filepath)[0]
        pickle_filepath = f'{filename}.pkl'

        funcs += [data_to_pickle]
        args += [(data_filepath, pickle_filepath)]
        kwargs += [dict()]

results = parallel(funcs, args, kwargs, key=lambda args: args[0], max_workers=3)

errors = []
original, pickled = 0, 0
for t, tpl in enumerate(results):
    data_filepath, result = tpl
    print(f'{t+1} / {len(results)} - "{os.path.relpath(data_filepath, PROJECT_DIRECTORY)}"')
    if isinstance(result, Exception):
        errors.append(data_filepath)
    else:
        original += os.path.getsize(data_filepath)
        pickled += os.path.getsize(pickle_filepath)

print(f'saved {((original - pickled) / original * 100):0.3f}% from {original / 1024**3:0.3f}GB to {pickled / 1024**3:0.3f}GB')
for error in errors:
    print(f'  - "{error}"')


1 / 27 - FAILURE! c:\Users\chris\src\sjsu.CMPE-257.ml-social-bot-detection\ignoreme\data\cresci-17\datasets_full.csv\datasets_full.csv\genuine_accounts.csv\genuine_accounts.csv\tweets.csv: Error tokenizing data. C error: Expected 26 fields in line 8, saw 27

2 / 27 - FAILURE! c:\Users\chris\src\sjsu.CMPE-257.ml-social-bot-detection\ignoreme\data\cresci-17\datasets_full.csv\datasets_full.csv\social_spambots_2.csv\social_spambots_2.csv\tweets.csv: 'utf-8' codec can't decode byte 0x97 in position 10: invalid start byte
3 / 27 - FAILURE! c:\Users\chris\src\sjsu.CMPE-257.ml-social-bot-detection\ignoreme\data\cresci-17\datasets_full.csv\datasets_full.csv\fake_followers.csv\fake_followers.csv\tweets.csv: 'utf-8' codec can't decode byte 0xf4 in position 9: invalid continuation byte
4 / 27 - FAILURE! c:\Users\chris\src\sjsu.CMPE-257.ml-social-bot-detection\ignoreme\data\cresci-17\datasets_full.csv\datasets_full.csv\traditional_spambots_1.csv\traditional_spambots_1.csv\tweets.csv: 'utf-8' codec 

5 / 27 - SUCCESS! c:\Users\chris\src\sjsu.CMPE-257.ml-social-bot-detection\ignoreme\data\cresci-17\datasets_full.csv\datasets_full.csv\genuine_accounts.csv\genuine_accounts.csv\users.csv: [('lang', 'category'), ('default_profile', 'category'), ('default_profile_image', 'category'), ('geo_enabled', 'category'), ('profile_use_background_image', 'category'), ('profile_background_tile', 'category'), ('utc_offset', 'category'), ('is_translator', 'category'), ('follow_request_sent', 'datetime'), ('follow_request_sent', 'category'), ('protected', 'category'), ('verified', 'category'), ('notifications', 'datetime'), ('notifications', 'category'), ('contributors_enabled', 'datetime'), ('contributors_enabled', 'category'), ('following', 'datetime'), ('following', 'category'), ('created_at', 'datetime'), ('timestamp', 'datetime'), ('crawled_at', 'datetime'), ('updated', 'datetime'), ('test_set_1', 'category'), ('test_set_2', 'category')]
6 / 27 - SUCCESS! c:\Users\chris\src\sjsu.CMPE-257.ml-socia

17 / 27 - FAILURE! c:\Users\chris\src\sjsu.CMPE-257.ml-social-bot-detection\ignoreme\data\cresci-17\datasets_full.csv\datasets_full.csv\social_spambots_1.csv\social_spambots_1.csv\tweets.csv: 'utf-8' codec can't decode byte 0xd2 in position 9: invalid continuation byte
18 / 27 - FAILURE! c:\Users\chris\src\sjsu.CMPE-257.ml-social-bot-detection\ignoreme\data\cresci-17\datasets_full.csv\datasets_full.csv\social_spambots_3.csv\social_spambots_3.csv\tweets.csv: 'utf-8' codec can't decode byte 0xf6 in position 9: invalid start byte


# Appendix


## Changelog
- 2026-11-13 16:32 - initial commit
